# 공용 학습 노트북 (Colab · Kaggle · VS Code 원격)

- **노트북은 이것 하나만 씁니다.** 실험마다 복사하지 않고, 셀 1의 `BRANCH`, `EXP_YAML` 두 줄만 바꾸세요. 바꾼 내용은 커밋하지 마세요.
- **학습은 마지막 셀의 `scripts/train.py`가 합니다.** 그 앞의 셀은 세션마다 반복해야 하는 준비(clone, `.env` 설정, 설치, 데이터)입니다.
- 새 세션이든 같은 세션이든 **Run All**하면 됩니다. 이미 끝난 준비는 빠르게 지나갑니다.
- 원격세션(Colab, Kaggle VM)에서는 GitHub의 브랜치를 clone해서 실행합니다. **실험 YAML을 먼저 push해놓고 실행하세요.**

| 실행 방식 | 실행 전에 할 일 |
| -- | -- |
| 브라우저 Colab | 런타임 → 런타임 유형 변경 → T4 GPU. 처음 한 번: 내 `.env`를 내 드라이브의 `MyDrive/.secret/pill-object-detection.env`로 올리기(공유 폴더 금지) |
| VS Code + Colab 확장 | Select Kernel → Colab → New Colab Server(이미 있으면 있는 거 사용) → GPU T4. Drive 연결을 허용하면 브라우저 Colab과 같은 `.env`를 읽음 |
| 브라우저 Kaggle | Settings → Accelerator **GPU T4 x2** (P100 금지), Internet On, Add-ons → Secrets에 `.env`와 같은 이름으로 `WANDB_API_KEY`, `WANDB_ENTITY`, Add Input → 팀 원본 데이터셋 |
| VS Code + Kaggle Jupyter Server | 위 Kaggle 설정을 웹에서 먼저 한 뒤, Run → Kaggle Jupyter Server의 VS Code Compatible URL을 Select Kernel → Existing Jupyter Server에 붙여 넣기 |

> 키와 개인 경로는 셀에 직접 쓰지 않고 `.env`에 둡니다. 
> 
> 커밋할 때 출력은 자동으로 지워집니다(nbstripout). 단, Colab의 **'GitHub에 사본 저장'으로 커밋하지 마세요.** 이 방법은 필터를 거치지 않아 출력이 그대로 올라갑니다.

In [ ]:
# ── 실험마다 바꾸는 값 ──
BRANCH = "main"                  # 실험 YAML을 push한 브랜치. 예) "exp/yolov8s-mosaic"
EXP_YAML = "configs/smoke.yaml"  # 예) "configs/exp/yolov8s_640.yaml"

# W&B 키·팀 이름과 Drive 원본 위치는 여기에 쓰지 않습니다. 아래 '설정 불러오기' 셀이 .env에서 읽습니다.

In [ ]:
# 실행 환경 확인
import os
import subprocess
import sys
from pathlib import Path


def sh(cmd):
    """셸 명령을 실행해 출력을 보여 주고, 실패하면 노트북 실행을 멈춘다."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"명령이 실패했습니다 (종료 코드 {proc.returncode}): {cmd}")


if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    ENV, REPO = "colab", Path("/content/pill-object-detection")
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    # /kaggle/working은 버전 저장 때 함께 저장되고 20GB 제한이 있어서, 레포와 .venv는 그 밖에 둔다
    ENV, REPO = "kaggle", Path("/kaggle/tmp/pill-object-detection")
else:
    ENV = "local"
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.environ["UV_FROZEN"] = "1"        # uv.lock에 적힌 버전 그대로 설치
os.environ["UV_LINK_MODE"] = "copy"  # uv 캐시와 .venv가 다른 디스크여도 경고 없이 설치
print("실행 환경:", ENV, "| 레포 위치:", REPO)

In [ ]:
# GPU 확인: T4가 아니면 멈춘다 (로컬은 건너뜀)
if ENV != "local":
    sh("nvidia-smi")  # 오른쪽 위 CUDA Version이 13.0 이상이어야 우리 torch(cu130)가 GPU를 쓸 수 있다
    gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout
    if "T4" not in gpu:
        raise RuntimeError("T4 GPU가 아닙니다. Colab: 런타임 → 런타임 유형 변경 → T4 GPU / "
                           "Kaggle: Settings → Accelerator → GPU T4 x2 (P100은 쓸 수 없음)")

In [ ]:
# 레포 받기: 원격에는 로컬 파일이 없으므로 GitHub에서 BRANCH를 받는다
REPO_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"
if ENV != "local":
    if not (REPO / ".git").exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        sh(f"git clone -b {BRANCH} {REPO_URL} {REPO}")
    else:  # 같은 세션에서 다시 실행: 원격 BRANCH의 최신 상태로 맞춤
        sh(f"git -C {REPO} fetch origin {BRANCH} && git -C {REPO} checkout -f -B {BRANCH} origin/{BRANCH}")
os.chdir(REPO)
sh("git log --oneline -1")

In [ ]:
# 설정 불러오기: 키와 개인 경로를 .env에서 읽어 환경 변수로 넣는다. 값은 출력하지 않는다.
#   로컬: 레포 루트 .env / Colab: 내 드라이브의 MyDrive/.secret/pill-object-detection.env / Kaggle: Add-ons → Secrets
# Drive 연결 창과 키 입력창이 이 셀에서 뜨므로, 오래 걸리는 설치보다 먼저 실행한다.
import getpass

DRIVE_ENV = Path("/content/drive/MyDrive/.secret/pill-object-detection.env")


def read_env_file(path):
    """KEY=VALUE 줄을 환경 변수로 넣는다. 이미 있는 환경 변수는 덮어쓰지 않는다 (python-dotenv와 같은 규칙)."""
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = (part.strip() for part in line.removeprefix("export ").split("=", 1))
        if value[:1] in ('"', "'"):  # 따옴표로 감싼 값
            value = value[1:].split(value[0], 1)[0]
        else:  # 줄 끝 주석 제거
            value = value.split(" #", 1)[0].strip()
        if value and not os.environ.get(key):
            os.environ[key] = value


def read_kaggle_secret(name):
    """Kaggle에는 .env를 둘 곳이 없어서 같은 이름의 Secret을 읽는다."""
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ[name] = UserSecretsClient().get_secret(name)
    except Exception:  # Secret이 없거나 노트북에 attach하지 않음
        pass


if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")  # 원본 데이터도 Drive에서 읽는다
for env_file in (REPO / ".env", DRIVE_ENV):
    if env_file.exists():
        read_env_file(env_file)
        print("읽은 파일:", env_file)
if ENV == "kaggle":
    for name in ("WANDB_API_KEY", "WANDB_ENTITY"):
        if not os.environ.get(name):
            read_kaggle_secret(name)

if ENV == "colab" and not os.environ.get("DRIVE_RAW"):
    raise RuntimeError(f"DRIVE_RAW가 없습니다. 로컬 .env에 DRIVE_RAW=<Drive의 원본 zip 또는 폴더 경로>를 적고, "
                       f"내 드라이브의 {DRIVE_ENV.relative_to('/content/drive')}로 다시 올리세요.")
if os.environ.get("WANDB_MODE") in ("offline", "disabled"):
    print("WANDB_MODE =", os.environ["WANDB_MODE"], "→ W&B 키 없이 진행")
elif not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("WANDB_API_KEY를 붙여 넣으세요 (화면에 보이지 않음): ").strip()
print("WANDB_API_KEY:", "설정됨" if os.environ.get("WANDB_API_KEY") else "없음")
print("WANDB_ENTITY:", "설정됨" if os.environ.get("WANDB_ENTITY") else "없음 → 개인 계정에 기록")

In [ ]:
# 패키지 설치: 팀원 모두 uv.lock에 적힌 같은 버전을 쓴다 (처음 세션은 torch를 받느라 몇 분 걸림)
if ENV != "local":
    sh("pip install -q uv")  # uv 자체를 설치할 때만 pip을 쓴다. 프로젝트 패키지는 모두 uv sync로 설치
sh("uv sync")
if ENV != "local":
    sh("""uv run python - <<'EOF'
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit("GPU를 쓸 수 없습니다. nvidia-smi의 CUDA Version이 13.0보다 낮은지 확인하세요.")
torch.zeros(1).cuda()  # 실제로 GPU에서 계산되는지 확인 (지원하지 않는 GPU면 여기서 실패)
print("GPU", torch.cuda.get_device_name(0), "| 지원 아키텍처", torch.cuda.get_arch_list())
EOF""")

In [ ]:
# 원본 데이터 배치 → YOLO 형식으로 변환
if ENV == "colab":
    drive_raw = os.environ["DRIVE_RAW"]  # 설정 셀에서 Drive 연결과 값 확인을 마침
    if drive_raw.endswith(".zip"):
        if not Path("/content/raw").exists():  # Drive에서 바로 읽으면 느려서 VM 디스크에 푼다
            sh(f'unzip -q "{drive_raw}" -d /content/raw')
        search_root = Path("/content/raw")
    else:
        search_root = Path(drive_raw)
elif ENV == "kaggle":
    search_root = Path("/kaggle/input")  # Add Input으로 붙인 데이터셋이 이 아래에 있다
else:
    search_root = REPO / "data" / "raw"

found = sorted(search_root.rglob("train_annotations"))
if not found:
    raise RuntimeError(f"{search_root} 아래에서 train_annotations 폴더를 찾지 못했습니다. "
                       "Colab은 .env의 DRIVE_RAW, Kaggle은 Add Input의 데이터셋을 확인하세요.")
RAW_DIR = found[0].parent
print("원본 데이터:", RAW_DIR)
sh(f'uv run python scripts/prepare_data.py --raw-dir "{RAW_DIR}"')

In [ ]:
# 학습
!uv run python scripts/train.py {EXP_YAML}

## 결과 확인
- 위 출력 마지막 부분의 W&B 링크에서 epoch별 loss, mAP, **mAP75-95(대회 지표)** 를 확인하세요.
- 원격 세션이 끝나면 `runs/`는 사라지지만 W&B의 기록과 best.pt(Artifacts 탭)는 남습니다.
- Kaggle 세션을 사용할 때, 몇 시간 걸리는 학습은 **Save Version → Save & Run All**로 돌리면 브라우저를 닫아도 계속됩니다.
- PR Description에 W&B run 링크와 mAP75-95를 적어주세요.